<a href="https://colab.research.google.com/github/assen-v/llm-gpu-benchmark/blob/main/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi


Mon Oct 12 14:09:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   60C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q transformers accelerate bitsandbytes datasets

In [6]:
import torch, time, re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))

name = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
tok.padding_side = "left"

2.11.0+cu130 True Tesla T4


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [9]:
ids = tok("What is the capital of France?")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))
print(tok.decode(ids))

ids = tok("Quantization of transformers is unbelievably useful")["input_ids"]
print(ids)
print(tok.convert_ids_to_tokens(ids))

[3838, 374, 279, 6722, 315, 9625, 30]
['What', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof', 'ĠFrance', '?']
What is the capital of France?
[44220, 2022, 315, 86870, 374, 38937, 88134, 5390]
['Quant', 'ization', 'Ġof', 'Ġtransformers', 'Ġis', 'Ġunbelie', 'vably', 'Ġuseful']


In [10]:
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16).to("cuda")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [11]:
!nvidia-smi

Mon Oct 12 15:09:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |    3239MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----